# Cross-Industry Accelerator — Load Lakehouse
### Auto-Load Dimension & Batch Fact Tables into Lakehouse Delta Tables

This notebook loads all **dimension tables** and **batch fact tables** into Fabric Lakehouse
as Delta tables, plus all **event/streaming tables** into the Eventhouse via KQL ingestion.

**What it does:**
1. Reads industry configuration from `00_Industry_Config`
2. Loads `dim_*` + `fact_*` (batch) tables → Lakehouse as Delta tables
3. Loads `fact_*` (event) + `stream_*` tables → Eventhouse KQL tables
4. Generates a load summary with row counts and status

> **Prerequisites:**
> 1. Run `01_Data_Ingestion.ipynb` to validate all CSV sources
> 2. Attach a **Lakehouse** to this notebook
> 3. For Eventhouse loading: fill in `EVENTHOUSE_CLUSTER_URI` and `EVENTHOUSE_DATABASE` in `00_Industry_Config`

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# RUN CONFIGURATION NOTEBOOK
# ════════════════════════════════════════════════════════════════════════

In [ ]:
%run 00_Industry_Config

In [ ]:
%run Pipeline_Logger

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# START PIPELINE STEP TRACKING
# ════════════════════════════════════════════════════════════════════════

try:
    pipeline_step_start("02_Load_Lakehouse", "Load dim/fact → Lakehouse Delta; events → Eventhouse")
except NameError:
    pass

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD DIMENSION TABLES → LAKEHOUSE DELTA (ZT-hardened)
# ════════════════════════════════════════════════════════════════════════
# Fabric Lakehouse Spark writes to the default schema. Tables automatically
# appear under 'dbo' in the SQL analytics endpoint — no schema prefix needed.
# Do NOT use CREATE SCHEMA/DATABASE or schema-qualified names in saveAsTable.

import pandas as pd

results = []

print(f"Loading {len(DIM_TABLES)} dimension tables into Lakehouse...\n")

for table_name in DIM_TABLES:
    csv_path = f"{_FS_CSV_PATH}/{table_name}.csv"
    try:
        df = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(csv_path))
        row_count = df.count()
        df.write.mode("overwrite").saveAsTable(table_name)
        results.append((table_name, "Dimension", row_count, len(df.columns), "✓"))
        log_audit_event("LAKEHOUSE_LOAD", table_name, f"{row_count} rows written")
        print(f"  ✓ {table_name:<45} {row_count:>6} rows  {len(df.columns):>3} cols")
    except Exception as e:
        results.append((table_name, "Dimension", 0, 0, f"✗ {e}"))
        log_audit_event("LAKEHOUSE_LOAD", table_name, str(e), "ERROR")
        print(f"  ✗ {table_name:<45} ERROR: {e}")

print(f"\n✅ Dimension tables loaded: {sum(1 for r in results if r[4] == '✓')}/{len(DIM_TABLES)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD BATCH FACT TABLES → LAKEHOUSE DELTA (ZT-hardened)
# ════════════════════════════════════════════════════════════════════════

print(f"Loading {len(FACT_TABLES_BATCH)} batch fact tables into Lakehouse...\n")

for table_name in FACT_TABLES_BATCH:
    csv_path = f"{_FS_CSV_PATH}/{table_name}.csv"
    try:
        df = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(csv_path))
        row_count = df.count()
        df.write.mode("overwrite").saveAsTable(table_name)
        results.append((table_name, "Fact (Batch)", row_count, len(df.columns), "✓"))
        log_audit_event("LAKEHOUSE_LOAD", table_name, f"{row_count} rows written")
        print(f"  ✓ {table_name:<45} {row_count:>6} rows  {len(df.columns):>3} cols")
    except Exception as e:
        results.append((table_name, "Fact (Batch)", 0, 0, f"✗ {e}"))
        log_audit_event("LAKEHOUSE_LOAD", table_name, str(e), "ERROR")
        print(f"  ✗ {table_name:<45} ERROR: {e}")

print(f"\n✅ Batch fact tables loaded: {sum(1 for r in results if r[1] == 'Fact (Batch)' and r[4] == '✓')}/{len(FACT_TABLES_BATCH)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD EVENT & STREAMING TABLES → EVENTHOUSE (KQL) — Direct REST API
# ════════════════════════════════════════════════════════════════════════
# Uses the Kusto REST API directly (more reliable than the Spark connector
# for table creation + small dataset ingestion in Fabric).

import requests as req_lib
import json

if not EVENTHOUSE_CLUSTER_URI or not EVENTHOUSE_DATABASE:
    print("Eventhouse not configured — skipping event/streaming table loading.")
    print("  Set EVENTHOUSE_CLUSTER_URI and EVENTHOUSE_DATABASE in 00_Industry_Config to enable.")
else:
    # ZT: Validate Eventhouse URI
    from urllib.parse import urlparse
    parsed_eh = urlparse(EVENTHOUSE_CLUSTER_URI)
    if not parsed_eh.hostname or not parsed_eh.hostname.endswith('.kusto.fabric.microsoft.com'):
        raise ValueError(f"ZT: Invalid Eventhouse URI: {EVENTHOUSE_CLUSTER_URI}")
    log_audit_event("EVENTHOUSE_CONNECT", EVENTHOUSE_CLUSTER_URI, "URI validated")

    access_token = notebookutils.credentials.getToken(EVENTHOUSE_CLUSTER_URI)

    def kql_mgmt(cmd):
        """Execute a KQL management command via REST API."""
        resp = req_lib.post(
            f"{EVENTHOUSE_CLUSTER_URI}/v1/rest/mgmt",
            headers={
                "Authorization": f"Bearer {access_token}",
                "Content-Type": "application/json",
            },
            json={"db": EVENTHOUSE_DATABASE, "csl": cmd},
        )
        if resp.status_code != 200:
            raise RuntimeError(f"KQL mgmt failed ({resp.status_code}): {resp.text[:300]}")
        return resp.json()

    def kql_query(query):
        """Execute a KQL query via REST API."""
        resp = req_lib.post(
            f"{EVENTHOUSE_CLUSTER_URI}/v1/rest/query",
            headers={
                "Authorization": f"Bearer {access_token}",
                "Content-Type": "application/json",
            },
            json={"db": EVENTHOUSE_DATABASE, "csl": query},
        )
        if resp.status_code != 200:
            raise RuntimeError(f"KQL query failed ({resp.status_code}): {resp.text[:300]}")
        return resp.json()

    # Spark type → KQL type mapping
    SPARK_TO_KQL = {
        "StringType": "string", "IntegerType": "int", "LongType": "long",
        "FloatType": "real", "DoubleType": "real", "BooleanType": "bool",
        "DateType": "datetime", "TimestampType": "datetime",
    }

    print(f"Loading {len(EVENTHOUSE_TABLES)} tables into Eventhouse ({EVENTHOUSE_DATABASE})...\n")

    for table_name in EVENTHOUSE_TABLES:
        csv_path = f"{_FS_CSV_PATH}/{table_name}.csv"
        category = "Streaming" if table_name.startswith("stream_") else "Fact (Event)"
        kql_table = table_name.replace("fact_", "").replace("stream_", "")

        try:
            sanitize_identifier(kql_table)
        except ValueError as e:
            log_audit_event("KQL_TABLE_REJECTED", kql_table, str(e), "BLOCKED")
            results.append((table_name, category, 0, 0, f"ZT: {e}"))
            continue

        try:
            # Read CSV via Spark
            df = (spark.read
                  .option("header", True)
                  .option("inferSchema", True)
                  .csv(csv_path))

            # Build KQL column definitions from Spark schema
            col_defs = []
            for field in df.schema.fields:
                kql_type = SPARK_TO_KQL.get(str(type(field.dataType).__name__), "string")
                col_defs.append(f"['{field.name}']:{kql_type}")

            # Create table via KQL management command
            create_cmd = f".create-merge table {kql_table} ({', '.join(col_defs)})"
            kql_mgmt(create_cmd)

            # Convert to CSV string for inline ingestion
            pdf = df.toPandas()
            csv_lines = []
            for _, row in pdf.iterrows():
                vals = []
                for v in row.values:
                    if v is None:
                        vals.append("")
                    elif hasattr(v, 'item'):
                        vals.append(str(v.item()))
                    else:
                        s = str(v).replace('"', '""')
                        if ',' in s or '"' in s or '\n' in s:
                            s = f'"{s}"'
                        vals.append(s)
                csv_lines.append(",".join(vals))

            # Ingest inline (immediate, no batching delay)
            csv_data = "\n".join(csv_lines)
            ingest_cmd = f".ingest inline into table {kql_table} <|\n{csv_data}"
            kql_mgmt(ingest_cmd)

            row_count = len(pdf)
            results.append((table_name, category, row_count, len(pdf.columns), "ok"))
            log_audit_event("EVENTHOUSE_LOAD", kql_table, f"{row_count} rows ingested")
            print(f"  ok {table_name:<35} -> {kql_table:<30} {row_count:>6} rows  {len(pdf.columns):>3} cols")

        except Exception as e:
            results.append((table_name, category, 0, 0, f"ERR: {e}"))
            log_audit_event("EVENTHOUSE_LOAD", kql_table, str(e), "ERROR")
            print(f"  ERR {table_name:<35} -> {kql_table:<30} {str(e)[:100]}")

    eh_ok = sum(1 for r in results if r[4] == 'ok')
    print(f"\nEventhouse tables loaded: {eh_ok}/{len(EVENTHOUSE_TABLES)}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# VERIFY EVENTHOUSE DATA — Diagnostic cell
# ════════════════════════════════════════════════════════════════════════
# Reads back from the Eventhouse to confirm data landed.

print(f"Checking Eventhouse: {EVENTHOUSE_CLUSTER_URI}")
print(f"Database: {EVENTHOUSE_DATABASE}")
print(f"Tables expected: {len(EVENTHOUSE_TABLES)}\n")

if not EVENTHOUSE_CLUSTER_URI:
    print("ERROR: EVENTHOUSE_CLUSTER_URI is empty!")
else:
    access_token = notebookutils.credentials.getToken(EVENTHOUSE_CLUSTER_URI)

    # Try reading back one table to verify
    test_table = EVENTHOUSE_TABLES[0].replace("fact_", "").replace("stream_", "")
    print(f"Reading back table '{test_table}' from Eventhouse...\n")

    try:
        verify_df = (spark.read
            .format("com.microsoft.kusto.spark.synapse.datasource")
            .option("kustoCluster", EVENTHOUSE_CLUSTER_URI)
            .option("kustoDatabase", EVENTHOUSE_DATABASE)
            .option("kustoQuery", f"{test_table} | take 5")
            .option("accessToken", access_token)
            .load())
        print(f"SUCCESS: {test_table} has data:")
        verify_df.show(5, truncate=False)
    except Exception as e:
        error_msg = str(e)
        print(f"FAILED to read {test_table}: {error_msg[:500]}")

        if "does not exist" in error_msg.lower() or "not found" in error_msg.lower():
            print("\nTable does not exist. The Kusto Spark write may have silently failed.")
            print("Trying to list all tables in the database...\n")

            try:
                tables_df = (spark.read
                    .format("com.microsoft.kusto.spark.synapse.datasource")
                    .option("kustoCluster", EVENTHOUSE_CLUSTER_URI)
                    .option("kustoDatabase", EVENTHOUSE_DATABASE)
                    .option("kustoQuery", ".show tables | project TableName, TotalRowCount")
                    .option("accessToken", access_token)
                    .load())
                print("Tables in database:")
                tables_df.show(20, truncate=False)
            except Exception as e2:
                print(f"Could not list tables: {e2}")
                print("\nThis suggests the database itself may not be accessible.")
                print(f"Verify in Fabric portal that '{EVENTHOUSE_DATABASE}' exists")
                print(f"under Eventhouse at: {EVENTHOUSE_CLUSTER_URI}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CREATE EVENTHOUSE + KQL DATABASE + EVENTSTREAM VIA FABRIC REST API
# ════════════════════════════════════════════════════════════════════════
# Automates creation of Eventhouse, KQL Database, and Eventstream items.
# The Eventstream → Eventhouse routing must still be configured once in
# the Fabric UI (no public API yet), but this cell creates all items
# and prints step-by-step instructions for the final wiring.

import requests, json, time
import sempy.fabric as fabric

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}
BASE_URL = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

def _create_item_if_not_exists(display_name, item_type):
    """Create a Fabric workspace item if it doesn't already exist."""
    items = fabric.list_items()
    matches = items[(items["Type"] == item_type) & (items["Display Name"] == display_name)]
    if not matches.empty:
        item_id = matches.iloc[0].Id
        print(f"  OK {item_type} '{display_name}' already exists -> {item_id}")
        return item_id

    print(f"  ... Creating {item_type} '{display_name}'...")
    body = {"displayName": display_name, "type": item_type}
    resp = requests.post(f"{BASE_URL}/items", headers=headers, json=body)

    if resp.status_code == 202:
        op_url = resp.headers.get("Location", "")
        for _ in range(30):
            time.sleep(5)
            poll = requests.get(op_url, headers=headers)
            if poll.status_code == 200 and poll.json().get("status") == "Succeeded":
                break
        items = fabric.list_items()
        matches = items[(items["Type"] == item_type) & (items["Display Name"] == display_name)]
        if not matches.empty:
            item_id = matches.iloc[0].Id
            print(f"  OK {item_type} '{display_name}' created -> {item_id}")
            return item_id
    elif resp.status_code in (200, 201):
        item_id = resp.json().get("id", "")
        print(f"  OK {item_type} '{display_name}' created -> {item_id}")
        return item_id

    print(f"  FAIL: Could not create {item_type}: {resp.status_code} - {resp.text}")
    return None

# 1. Create Eventhouse
eh_name = EVENTHOUSE_NAME
eh_id = _create_item_if_not_exists(eh_name, "Eventhouse")

# 2. Create KQL Database inside the Eventhouse
kql_db_name = EVENTHOUSE_DATABASE
kql_db_id = None
if eh_id:
    kql_db_id = _create_item_if_not_exists(kql_db_name, "KQLDatabase")

# 3. Create Eventstream
es_name = EVENTSTREAM_NAME
es_id = _create_item_if_not_exists(es_name, "Eventstream")

# Print KQL tables loaded
eh_tables = [t.replace("fact_", "").replace("stream_", "") for t in EVENTHOUSE_TABLES]

print(f"\n{'='*70}")
print(f"EVENTHOUSE SETUP COMPLETE - {INDUSTRY.upper()}")
print(f"{'='*70}")
print(f"  Eventhouse:   {eh_name} ({eh_id})")
print(f"  KQL Database: {kql_db_name} ({kql_db_id})")
print(f"  Eventstream:  {es_name} ({es_id})")
print(f"  KQL Tables:   {len(eh_tables)} loaded")
for t in eh_tables:
    print(f"    - {t}")

print(f"\n{'~'*70}")
print(f"ONE-TIME MANUAL STEP: Configure Eventstream Routing")
print(f"{'~'*70}")
print(f"Eventstream-to-Eventhouse routing cannot yet be configured via API.")
print(f"Do this ONCE in the Fabric portal:\n")
print(f"1. Open Eventstream '{es_name}' in your workspace")
print(f"2. Add Source -> Custom Endpoint")
print(f"   Copy the Event Hub connection string for your simulator/.env")
print(f"3. Add Destination -> Eventhouse")
print(f"   Eventhouse:   {eh_name}")
print(f"   KQL Database: {kql_db_name}")
print(f"   Route to individual tables:")
for t in eh_tables:
    print(f"      - {t}")
print(f"4. Data format: JSON")
print(f"5. Click Activate on each destination")
print(f"\nVerify with KQL after configuring:")
for t in eh_tables[:3]:
    print(f"  {t} | count")
if len(eh_tables) > 3:
    print(f"  ...")
print(f"{'~'*70}")

log_audit_event("EVENTHOUSE_SETUP", eh_name,
    f"Eventhouse + KQL DB + Eventstream created. {len(eh_tables)} tables loaded.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LIVE STREAMING SIMULATOR → EVENTSTREAM → EVENTHOUSE
# ════════════════════════════════════════════════════════════════════════
# Replays stream_* CSV data as live JSON events into the Eventstream's
# Custom Endpoint using the Event Hub protocol.
#
# BEFORE RUNNING: In the Fabric portal, open your Eventstream and:
#   1. Add Source → Custom Endpoint (if not already done)
#   2. Click the Custom Endpoint node → Keys
#   3. Copy the "Connection string" and "Event Hub name"
#   4. Paste them below as EVENT_HUB_CONNECTION_STRING and EVENT_HUB_NAME
#   5. Add Destination → Eventhouse (Direct ingestion) → pick table
# ════════════════════════════════════════════════════════════════════════

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "azure-eventhub", "-q"])

import csv, json, random, time as time_mod, os
import datetime as dt
from azure.eventhub import EventHubProducerClient, EventData

# ── CONFIGURATION — paste your Eventstream Custom Endpoint credentials ──
EVENT_HUB_CONNECTION_STRING = ""  # Endpoint=sb://...;SharedAccessKeyName=...;SharedAccessKey=...;EntityPath=...
EVENT_HUB_NAME = ""               # e.g. es_xxxxxxxx

# Simulation settings
INTERVAL = 0.3       # seconds between batches
LOOPS = 2            # how many times to replay the full dataset
BATCH_SIZE_MAX = 10  # max events per batch

# ── Validate credentials ────────────────────────────────────────────────
if not EVENT_HUB_CONNECTION_STRING or not EVENT_HUB_NAME:
    print("=" * 70)
    print("SETUP REQUIRED: Paste your Eventstream Custom Endpoint credentials")
    print("=" * 70)
    print()
    print("1. Open Eventstream in Fabric portal")
    print("2. Click Custom Endpoint source node -> Keys")
    print("3. Copy 'Connection string' and 'Event Hub name'")
    print("4. Paste them in this cell as:")
    print('   EVENT_HUB_CONNECTION_STRING = "Endpoint=sb://..."')
    print('   EVENT_HUB_NAME = "es_xxxxxxxx"')
    print()
    print("Then re-run this cell.")
    print("=" * 70)
else:
    # ── Load streaming CSVs from lakehouse ──────────────────────────────
    all_events = []
    for table_name in STREAM_TABLES:
        csv_path = f"{_FS_CSV_PATH}/{table_name}.csv"
        try:
            pdf = (spark.read
                   .option("header", True)
                   .option("inferSchema", True)
                   .csv(csv_path)
                   .toPandas())
            stream_type = table_name.replace("stream_", "")
            for _, row in pdf.iterrows():
                all_events.append((stream_type, row.to_dict()))
            print(f"  Loaded {len(pdf):>4} rows from {table_name}")
        except Exception as e:
            print(f"  WARN: Could not load {table_name}: {e}")

    if not all_events:
        print("No streaming data found. Check STREAM_TABLES and CSV paths.")
    else:
        print(f"\n  Total events per loop: {len(all_events)}")
        print(f"  Loops: {LOOPS}")
        print(f"  Interval: {INTERVAL}s between batches")
        print(f"  Sending to: {EVENT_HUB_NAME}\n")

        # ── Connect and send ────────────────────────────────────────────
        producer = EventHubProducerClient.from_connection_string(
            conn_str=EVENT_HUB_CONNECTION_STRING,
            eventhub_name=EVENT_HUB_NAME,
        )

        total_sent = 0
        try:
            for loop_num in range(1, LOOPS + 1):
                random.shuffle(all_events)
                now = dt.datetime.utcnow()
                i = 0
                print(f"-- Loop {loop_num}/{LOOPS} (base: {now.strftime('%H:%M:%S')}Z) --")

                while i < len(all_events):
                    batch = producer.create_batch()
                    batch_count = 0
                    batch_size = random.randint(1, BATCH_SIZE_MAX)

                    for j in range(i, min(i + batch_size, len(all_events))):
                        stream_type, row = all_events[j]
                        # Rebase timestamps to now
                        event_data = {}
                        for k, v in row.items():
                            if "timestamp" in k.lower() or k.lower() == "time":
                                try:
                                    orig = dt.datetime.fromisoformat(str(v).replace("Z", ""))
                                    rebased = now + dt.timedelta(seconds=total_sent * 0.1)
                                    v = rebased.strftime("%Y-%m-%dT%H:%M:%SZ")
                                except (ValueError, TypeError):
                                    pass
                            # Convert non-serializable types (numpy scalars)
                            if hasattr(v, 'item'):
                                v = v.item()
                            event_data[k] = v

                        envelope = {
                            "stream_type": stream_type,
                            "ingestion_time": dt.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
                            "data": event_data,
                        }
                        try:
                            batch.add(EventData(json.dumps(envelope, default=str)))
                            batch_count += 1
                        except ValueError:
                            break

                    producer.send_batch(batch)
                    total_sent += batch_count
                    i += batch_count

                    if total_sent % 50 == 0 or i >= len(all_events):
                        print(f"    Sent {total_sent} events total (last: {stream_type})")

                    time_mod.sleep(INTERVAL)

                print(f"  Loop {loop_num} done ({len(all_events)} events)\n")

        except KeyboardInterrupt:
            print(f"\n  Interrupted after {total_sent} events.")
        finally:
            producer.close()

        print(f"\nSimulation complete: {total_sent} events sent to Eventstream.")
        print(f"Verify in Eventhouse with KQL:")
        for t in [s.replace("stream_", "") for s in STREAM_TABLES[:3]]:
            print(f"  {t} | where ingestion_time() > ago(10m) | count")
        log_audit_event("STREAM_SIMULATOR", EVENTSTREAM_NAME,
            f"{total_sent} events sent in {LOOPS} loops")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# LOAD SUMMARY REPORT
# ════════════════════════════════════════════════════════════════════════

summary_df = pd.DataFrame(results, columns=["Table", "Category", "Rows", "Columns", "Status"])

print(f"\n{'═'*75}")
print(f"LAKEHOUSE LOAD SUMMARY — {INDUSTRY.upper()}")
print(f"{'═'*75}")
print(summary_df.to_string(index=False))
print(f"\n{'─'*75}")
print(f"Total rows loaded:    {summary_df[summary_df['Status']=='✓']['Rows'].sum():,}")
print(f"Successful tables:    {(summary_df['Status']=='✓').sum()}")
print(f"Failed tables:        {(summary_df['Status']!='✓').sum()}")
print(f"\n✅ Lakehouse + Eventhouse loading complete.")
print(f"   Next: Run 03_Load_Warehouse.ipynb to load the Fabric Data Warehouse.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# COMPLETE PIPELINE STEP & LOG RECORDS
# ════════════════════════════════════════════════════════════════════════

try:
    # Log record counts for each loaded table
    for table_name, category, row_count, col_count, status in results:
        if "SUCCESS" in status or "✓" in status:
            target = "Lakehouse" if category in ["dimension", "fact-batch"] else "Eventhouse"
            log_records_loaded("02_Load_Lakehouse", table_name, row_count, target=target)
    
    total_rows = sum(r[2] for r in results if len(r) > 2)
    pipeline_step_complete("02_Load_Lakehouse", f"Loaded {len(results)} tables with {total_rows:,} rows")
except NameError:
    pass